# 🏭 Sprint 2 — Visualização Operacional e Representação do Ativo
## FIAP × Forzy | Processamento de Linguagem Natural, Chatbots & Virtual Agents

---

### 👥 Grupo
| Nome | RM |
|---|---|
| Augusto Oliveira Codo de Sousa | RM562080 |
| Felipe de Oliveira Cabral | RM561720 |
| Gabriel Tonelli Avelino Dos Santos | RM564705 |
| Vinícius Adrian Siqueira de Oliveira | RM564962 |
| Sofia Bueris Netto de Souza | RM565818 |

---

### 🗺️ Pipeline da Sprint 2

```
[Corpus Sprint 1]
      │
      ▼
[Célula 0] Configuração de Caminhos & Validação de Estrutura
      │
      ▼
[Célula 1] Setup do Ambiente (sentence-transformers, FAISS, pandas)
      │
      ▼
[Célula 2] Digital Twin — Base de Ativos (DataFrame)
      │
      ▼
[Célula 3] Entregável 1 — Representação Vetorial + Indexação FAISS
      │
      ▼
[Célula 4] Entregável 2 — Motor de Busca Semântica + Normalização Terminológica
      │
      ▼
[Célula 5] Avaliação — Precisão e Recall sobre Consultas-Teste Manuais
      │
      ▼
[Célula 6] Entregável 3 — NLG Operacional: Templates + Documentação
      │
      ▼
[Célula 7] Relatório de Saída + Integração Sprint 1
```

| Módulo | Tecnologia | Justificativa |
|---|---|---|
| Embedding | `paraphrase-multilingual-MiniLM-L12-v2` | Multilingual PT-BR/EN, 384 dims, leve e preciso para domínio técnico |
| Vector DB | `FAISS IndexFlatL2` | Busca exata por similaridade, zero infraestrutura, determinístico |
| Normalização | `tabela_normalizacao.csv` (Sprint 1) | 790 variantes → termos canônicos, cobertura 100% do glossário |
| NLG | Template-based | Reprodutível, auditável, sem custo de LLM, controlável pelo operador |


## Célula 0 — Configuração de Caminhos

In [1]:
# ==============================================================================
# CÉLULA 0: CONFIGURAÇÃO DE CAMINHOS
# Detecta automaticamente se está rodando no Colab (após clone do repositório)
# ou localmente. Basta clonar o repo e executar — nenhuma edição necessária.
#
# Estrutura esperada do repositório:
#   sprint2/
#   |-- sprint2_pln_consulta.ipynb  <- este notebook
#   |-- inputs/                     <- glossario_tecnico_motores_v5.{csv,json}
#   |-- data/                       <- corpus_consolidado_v1.csv, corpus_motores_v1.{json,txt}
#   |-- tabelas/                    <- tabela_normalizacao.csv, stopwords_tecnicas.txt
#   |-- relatorios/                 <- gerado em runtime (relatórios, métricas)
#   |-- README.md
# ==============================================================================

import os, sys

# Detecta se está no Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Instrução única: monte o Drive e ajuste REPO_ROOT para onde clonou
    # Exemplo: !git clone https://github.com/SEU_USER/SEU_REPO.git
    REPO_ROOT = '/content/sprint2'
else:
    # Execução local: notebook está dentro da pasta sprint2/
    REPO_ROOT = os.path.dirname(os.path.abspath('sprint2_pln_consulta.ipynb'))

# Pastas do repositório
DIR_INPUTS     = os.path.join(REPO_ROOT, 'inputs')
DIR_DATA       = os.path.join(REPO_ROOT, 'data')
DIR_TABELAS    = os.path.join(REPO_ROOT, 'tabelas')
DIR_RELATORIOS = os.path.join(REPO_ROOT, 'relatorios')

# Artefatos de entrada (Sprint 1 — já presentes no repositório)
PATH_GLOSSARIO_JSON = os.path.join(DIR_INPUTS,  'glossario_tecnico_motores_v5.json')
PATH_GLOSSARIO_CSV  = os.path.join(DIR_INPUTS,  'glossario_tecnico_motores_v5.csv')
PATH_STOPWORDS      = os.path.join(DIR_TABELAS, 'stopwords_tecnicas.txt')
PATH_NORMALIZACAO   = os.path.join(DIR_TABELAS, 'tabela_normalizacao.csv')
PATH_CORPUS_JSON    = os.path.join(DIR_DATA,    'corpus_motores_v1.json')
PATH_CORPUS_TXT     = os.path.join(DIR_DATA,    'corpus_motores_v1.txt')
PATH_CORPUS_CSV     = os.path.join(DIR_DATA,    'corpus_consolidado_v1.csv')

# Artefatos de saída (gerados em runtime — pasta relatorios/)
PATH_STATS_JSON   = os.path.join(DIR_RELATORIOS, 'corpus_stats_sprint2.json')
PATH_METRICAS_CSV = os.path.join(DIR_RELATORIOS, 'metricas_avaliacao_sprint2.csv')
PATH_TEMPLATES_MD = os.path.join(DIR_RELATORIOS, 'templates_nlg_sprint2.md')

# Garante que relatorios/ existe (as demais são rastreadas pelo git)
os.makedirs(DIR_RELATORIOS, exist_ok=True)

# Validação
print(f'Repositório raiz : {REPO_ROOT}')
print(f'Ambiente         : {"Google Colab" if IN_COLAB else "Local"}')
print()
print('Validação das pastas:')
erros = []
for nome, path in [('inputs', DIR_INPUTS),('data', DIR_DATA),
                    ('tabelas', DIR_TABELAS),('relatorios', DIR_RELATORIOS)]:
    existe = os.path.isdir(path)
    print(f'  {"OK" if existe else "AUSENTE":>8}  {nome}/')
    if not existe: erros.append(nome)
print()
if erros:
    print(f'AVISO: pastas ausentes: {erros}')
    print('Verifique se clonou o repositório completo.')
else:
    print('Estrutura de pastas OK. Avance para a Célula 1.')
print()
print('Arquivos de entrada esperados:')
for label, path in [
    ('tabela_normalizacao.csv',           PATH_NORMALIZACAO),
    ('glossario_tecnico_motores_v5.csv',  PATH_GLOSSARIO_CSV),
    ('corpus_consolidado_v1.csv',         PATH_CORPUS_CSV),
]:
    status = 'ENCONTRADO' if os.path.isfile(path) else 'AUSENTE'
    print(f'  {status:>10}  {label}')


Repositório raiz : c:\Users\gabri\Downloads\sprint2_PLN\sprint2
Ambiente         : Local

Validação das pastas:
        OK  inputs/
        OK  data/
        OK  tabelas/
        OK  relatorios/

Estrutura de pastas OK. Avance para a Célula 1.

Arquivos de entrada esperados:
  ENCONTRADO  tabela_normalizacao.csv
  ENCONTRADO  glossario_tecnico_motores_v5.csv
  ENCONTRADO  corpus_consolidado_v1.csv


## Célula 1 — Setup do Ambiente

### 🔬 Justificativa da Escolha do Modelo de Embeddings

O modelo **`paraphrase-multilingual-MiniLM-L12-v2`** foi selecionado pelos seguintes critérios:

| Critério | Análise |
|---|---|
| **Domínio Técnico-Industrial** | Treinado com dados de 50+ idiomas incluindo textos técnicos; captura bem semântica de siglas (RPM, IP55, IEC) e termos compostos ("Motor de Indução Trifásico") |
| **Bilíngue PT-BR / EN** | O corpus da Sprint 1 tem distribuição `pt: 321 docs | en: 107 docs`; o modelo multilingual trata ambos sem pipeline separado |
| **Eficiência Computacional** | 384 dimensões, ~117M parâmetros, executa em CPU no Colab em < 2s para o corpus atual |
| **Alternativas Avaliadas** | `all-MiniLM-L6-v2` (só inglês, descartado); `paraphrase-multilingual-mpnet-base-v2` (768 dims, mais lento, sem ganho significativo para corpus pequeno); `bert-base-multilingual-cased` (não fine-tunado para similaridade semântica) |
| **Adequação ao Corpus** | Cobertura 100% do glossário v5 (107 termos presentes no corpus per `m5_cobertura`); vocabulário lematizado 2.914 tipos — dimensão compatível com MiniLM-L12 |


In [2]:
# ==============================================================================
# CÉLULA 1: SETUP DO AMBIENTE — À PROVA DE FALHAS
# ==============================================================================

%pip install --upgrade sentence-transformers faiss-cpu pandas numpy --quiet

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import faiss
import json, os
from sentence_transformers import SentenceTransformer
import sentence_transformers

print('─' * 60)
print('📦 VERSÕES INSTALADAS NO KERNEL')
print('─' * 60)
print(f'  pandas               : {pd.__version__}')
print(f'  numpy                : {np.__version__}')
print(f'  faiss                : {faiss.__version__}')
print(f'  sentence-transformers: {sentence_transformers.__version__}')
print('─' * 60)

print('\n⏳ Carregando modelo de embeddings...')
MODELO_NOME = 'paraphrase-multilingual-MiniLM-L12-v2'
modelo_embedding = SentenceTransformer(MODELO_NOME)

teste_dim = modelo_embedding.encode(['teste de sanidade']).shape[1]
assert teste_dim == 384, f'Dimensão inesperada: {teste_dim} (esperado: 384)'

print('\n' + '=' * 60)
print('✅ AMBIENTE PRONTO PARA A SPRINT 2')
print('=' * 60)
print(f'  Modelo    : {MODELO_NOME}')
print(f'  Dim. vetor: {teste_dim}')
print(f'  Kernel    : {__import__("sys").executable}')
print('=' * 60)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
────────────────────────────────────────────────────────────
📦 VERSÕES INSTALADAS NO KERNEL
────────────────────────────────────────────────────────────
  pandas               : 3.0.3
  numpy                : 2.4.6
  faiss                : 1.14.3
  sentence-transformers: 5.6.0
────────────────────────────────────────────────────────────

⏳ Carregando modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


✅ AMBIENTE PRONTO PARA A SPRINT 2
  Modelo    : paraphrase-multilingual-MiniLM-L12-v2
  Dim. vetor: 384
  Kernel    : c:\Users\gabri\AppData\Local\Programs\Python\Python314\python.exe


## Célula 2 — Digital Twin: Base de Ativos

In [ ]:
# ==============================================================================
# CÉLULA 2: MOCK DE ATIVOS DO DIGITAL TWIN
# Representa a base de cadastro de equipamentos que seria consultada
# pelo operador em produção. Cada linha = um ativo monitorado.
# ==============================================================================

dados_ativos = [
    {
        'tag': 'MT-042',
        'equipamento': 'Motor de Indução Trifásico WEG W22',
        'especificacao': '380V, 15kW, 1750 RPM, IP55, Classe F, Fator de Serviço 1.15',
        'localizacao': 'Área 3 - Bomba de Resfriamento Principal',
        'fabricante': 'WEG',
        'norma': 'IEC 60034 / NBR 17094'
    },
    {
        'tag': 'MT-088',
        'equipamento': 'Motor Inverter-Duty ABB',
        'especificacao': '440V, 55kW, 3500 RPM, IP66, Classe H, Rolamento Isolado',
        'localizacao': 'Área 1 - Exaustor do Forno',
        'fabricante': 'ABB',
        'norma': 'NEMA MG1 / IEC 60034'
    },
    {
        'tag': 'MT-102',
        'equipamento': 'Motor Síncrono Siemens',
        'especificacao': '13.8kV, 500kW, 1800 RPM, IP55, Alta Tensão',
        'localizacao': 'Área 2 - Compressor de Gás Natural',
        'fabricante': 'Siemens',
        'norma': 'IEC 60034'
    },
    {
        'tag': 'MT-015',
        'equipamento': 'Motor Monofásico Nidec',
        'especificacao': '220V, 1.5kW, 3600 RPM, IP44, Classe B',
        'localizacao': 'Área 3 - Bomba de Drenagem Secundária',
        'fabricante': 'Nidec',
        'norma': 'NBR 17094'
    },
    {
        'tag': 'MT-205',
        'equipamento': 'Motor de Indução Trifásico WEG HGF',
        'especificacao': '690V, 250kW, 1200 RPM, IP55, Rolamento Isolado, Alta Eficiência IE3',
        'localizacao': 'Área 4 - Moinho de Bolas',
        'fabricante': 'WEG',
        'norma': 'IEC 60034 / NBR 17094'
    },
    {
        'tag': 'MT-310',
        'equipamento': 'Motor de Indução Trifásico Siemens SIMOTICS',
        'especificacao': '380V, 22kW, 1450 RPM, IP55, Classe F, IE2',
        'localizacao': 'Área 2 - Bomba de Circulação de Água',
        'fabricante': 'Siemens',
        'norma': 'IEC 60034'
    },
    {
        'tag': 'MT-411',
        'equipamento': 'Motor à Prova de Explosão WEG Ex-d',
        'especificacao': '440V, 30kW, 1750 RPM, IP66, Classe F, Zona 1 ATEX',
        'localizacao': 'Área 5 - Sala de Bombas Químicas',
        'fabricante': 'WEG',
        'norma': 'IEC 60079 / NBR IEC 60079'
    },
]

df_ativos = pd.DataFrame(dados_ativos)

# String de contexto: campo que o modelo vai vetorizar 
# Concatenamos todos os campos textuais para enriquecer a representação semântica.
# Quanto mais contexto, melhor a recuperação por similaridade.
df_ativos['contexto_busca'] = df_ativos.apply(
    lambda x: (
        f"TAG: {x['tag']} | "
        f"Equipamento: {x['equipamento']} | "
        f"Especificações: {x['especificacao']} | "
        f"Localização: {x['localizacao']} | "
        f"Fabricante: {x['fabricante']} | "
        f"Norma: {x['norma']}"
    ),
    axis=1
)

print(f'✅ Base do Digital Twin carregada com {len(df_ativos)} ativos operacionais.')
print()
print('🔍 Exemplo de String de Contexto (entrada para o modelo de embeddings):')
print(f'→ {df_ativos["contexto_busca"].iloc[0]}')
print()
print(df_ativos[['tag','equipamento','localizacao']].to_string(index=False))


✅ Base do Digital Twin carregada com 7 ativos operacionais.

🔍 Exemplo de String de Contexto (entrada para o modelo de embeddings):
→ TAG: MT-042 | Equipamento: Motor de Indução Trifásico WEG W22 | Especificações: 380V, 15kW, 1750 RPM, IP55, Classe F, Fator de Serviço 1.15 | Localização: Área 3 - Bomba de Resfriamento Principal | Fabricante: WEG | Norma: IEC 60034 / NBR 17094

   tag                                 equipamento                              localizacao
MT-042          Motor de Indução Trifásico WEG W22 Área 3 - Bomba de Resfriamento Principal
MT-088                     Motor Inverter-Duty ABB               Área 1 - Exaustor do Forno
MT-102                      Motor Síncrono Siemens       Área 2 - Compressor de Gás Natural
MT-015                      Motor Monofásico Nidec    Área 3 - Bomba de Drenagem Secundária
MT-205          Motor de Indução Trifásico WEG HGF                 Área 4 - Moinho de Bolas
MT-310 Motor de Indução Trifásico Siemens SIMOTICS     Área 2 - Bomb

## Célula 3 — Entregável 1: Representação Vetorial e Indexação FAISS

**Decisão de arquitetura:** `IndexFlatL2` realiza busca exata (força bruta) no espaço L2,
garantindo recall perfeito para bases pequenas (< 10k vetores). Para escala industrial
(> 100k ativos), recomenda-se migrar para `IndexIVFFlat` com `nlist = sqrt(n)`.


In [4]:
# ==============================================================================
# CÉLULA 3: REPRESENTAÇÃO VETORIAL E INDEXAÇÃO FAISS
# Entregável 1: Gerar embeddings dos campos textuais e armazenar em índice FAISS
# ==============================================================================

print('⏳ Gerando embeddings semânticos dos ativos...')
print('   Modelo:', MODELO_NOME)
print()

# 1. Gerar embeddings a partir da coluna de contexto
embeddings_ativos = modelo_embedding.encode(
    df_ativos['contexto_busca'].tolist(),
    show_progress_bar=True
)

# 2. Conversão obrigatória para float32 (requisito do FAISS — backend C++)
embeddings_ativos = np.array(embeddings_ativos).astype('float32')

# 3. Dimensão do espaço vetorial
dimensao = embeddings_ativos.shape[1]

# 4. Criar índice FAISS com Distância L2 (Euclidiana)
#    IndexFlatL2 = busca exata, sem aproximação, determinístico
indice_faiss = faiss.IndexFlatL2(dimensao)

# 5. Adicionar embeddings ao índice
indice_faiss.add(embeddings_ativos)

# 6. Verificação de sanidade: self-retrieval deve retornar distância 0
q_test = embeddings_ativos[0:1]
D_test, I_test = indice_faiss.search(q_test, 1)
assert I_test[0][0] == 0, 'Erro: self-retrieval falhou!'
assert D_test[0][0] < 1e-5, f'Distância inesperada: {D_test[0][0]}'

# 7. Persistir métricas
stats_sprint2 = {
    'total_vetores'  : int(indice_faiss.ntotal),
    'dimensao'       : int(dimensao),
    'dtype'          : str(embeddings_ativos.dtype),
    'modelo'         : MODELO_NOME,
    'tipo_indice'    : 'IndexFlatL2',
    'metrica'        : 'L2 (Euclidiana)',
    'self_retrieval' : 'PASS',
    'data_indexacao' : pd.Timestamp.now().isoformat()
}
os.makedirs(DIR_RELATORIOS, exist_ok=True)
with open(PATH_STATS_JSON, 'w', encoding='utf-8') as f:
    json.dump(stats_sprint2, f, ensure_ascii=False, indent=2)

print()
print('=' * 60)
print('✅ Índice Vetorial FAISS criado e populado com sucesso!')
print(f'⚙️  Total de vetores indexados : {indice_faiss.ntotal}')
print(f'📐 Dimensão do espaço vetorial: {dimensao} dimensões')
print(f'🔍 Tipo de índice              : IndexFlatL2 (busca exata)')
print(f'✔️  Self-retrieval test         : PASS (distância ≈ 0)')
print(f'💾 Métricas salvas em          : {PATH_STATS_JSON}')
print('=' * 60)

# 8. Visualização das distâncias inter-ativos (matriz parcial)
print()
print('📊 Matriz de Distâncias L2 (primeiros 4 ativos × 4 ativos):')
print('─' * 60)
D_mat, _ = indice_faiss.search(embeddings_ativos[:4], 4)
tags_4 = df_ativos['tag'].values[:4]
header = f'{"":>8}' + ''.join(f'{t:>10}' for t in tags_4)
print(header)
for i, tag in enumerate(tags_4):
    row = f'{tag:>8}' + ''.join(f'{D_mat[i,j]:>10.3f}' for j in range(4))
    print(row)
print('─' * 60)
print('Interpretação: 0.000 = idêntico | valores maiores = menos similares')


⏳ Gerando embeddings semânticos dos ativos...
   Modelo: paraphrase-multilingual-MiniLM-L12-v2



Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Índice Vetorial FAISS criado e populado com sucesso!
⚙️  Total de vetores indexados : 7
📐 Dimensão do espaço vetorial: 384 dimensões
🔍 Tipo de índice              : IndexFlatL2 (busca exata)
✔️  Self-retrieval test         : PASS (distância ≈ 0)
💾 Métricas salvas em          : c:\Users\gabri\Downloads\sprint2_PLN\sprint2\relatorios\corpus_stats_sprint2.json

📊 Matriz de Distâncias L2 (primeiros 4 ativos × 4 ativos):
────────────────────────────────────────────────────────────
            MT-042    MT-088    MT-102    MT-015
  MT-042     0.000     2.664     3.000     3.135
  MT-088     0.000     3.409     3.884     3.923
  MT-102     0.000     3.236     4.269     4.687
  MT-015     0.000     3.135     3.733     3.884
────────────────────────────────────────────────────────────
Interpretação: 0.000 = idêntico | valores maiores = menos similares


## Célula 4 — Entregável 2: Motor de Busca Semântica

**Pipeline de recuperação:**

```
query bruta (jargão heterogêneo)
  |
  v  normalizar_query() — tabela_normalizacao.csv (790 variantes)
     backoff: trigrama -> bigrama -> unigrama
query normalizada
  |
  v  modelo_embedding.encode() — paraphrase-multilingual-MiniLM-L12-v2
vetor [1 x 384]
  |
  v  indice_faiss.search(top_k)
[(tag, ativo, dist_L2, score), ...]
```


In [5]:
df_norm = pd.read_csv(PATH_NORMALIZACAO, encoding='utf-8')
dic_norm = dict(zip(
    df_norm['variante'].str.lower().str.strip(),
    df_norm['termo_canonico'].str.strip()
))
print(f'Normalizacao: {len(dic_norm)} variantes')
print(f'  ac drive        -> {dic_norm.get("ac drive","N/A")}')
print(f'  bearing failure -> {dic_norm.get("bearing failure","N/A")}')
print(f'  oc fault        -> {dic_norm.get("oc fault","N/A")}')
print(f'  thermal overload-> {dic_norm.get("thermal overload","N/A")}')
print()


def normalizar_query(q):
    tokens = q.lower().strip().split()
    saida, subs = [], []
    i = 0
    while i < len(tokens):
        tri = ' '.join(tokens[i:i+3])
        bi  = ' '.join(tokens[i:i+2])
        uni = tokens[i]
        if i+3 <= len(tokens) and tri in dic_norm:
            c = dic_norm[tri]; saida.append(c); subs.append((tri,c)); i+=3
        elif i+2 <= len(tokens) and bi in dic_norm:
            c = dic_norm[bi];  saida.append(c); subs.append((bi,c));  i+=2
        elif uni in dic_norm:
            c = dic_norm[uni]; saida.append(c); subs.append((uni,c)); i+=1
        else:
            saida.append(uni); i+=1
    return ' '.join(saida), subs


def buscar_ativo(query, top_k=2, verbose=True):
    q_norm, subs = normalizar_query(query)
    if verbose:
        print('-' * 70)
        print(f'  Original   : "{query}"')
        print(f'  Normalizada: "{q_norm}"')
        for orig, can in subs:
            print(f'     -> "{orig}" -> "{can}"')
        print('-' * 70)
    q_vec = modelo_embedding.encode([q_norm], convert_to_numpy=True).astype('float32')
    D, I  = indice_faiss.search(q_vec, top_k)
    resultados = []
    for rank, (idx, dist) in enumerate(zip(I[0], D[0])):
        score = round(1/(1+float(dist)),4)
        ativo = df_ativos.iloc[idx].to_dict()
        ativo.update({'_rank':rank+1,'_dist':round(float(dist),4),'_score':score})
        resultados.append(ativo)
        if verbose:
            print(f'  [{rank+1}] {ativo["tag"]}  L2={dist:.4f} | Score={score:.4f}')
            print(f'      {ativo["localizacao"]}')
            print(f'      {ativo["equipamento"]}')
    if verbose: print()
    return resultados


print('DEMONSTRACAO DO MOTOR DE BUSCA')
print('=' * 70)
print()
buscar_ativo('motor da area 3', top_k=2)
buscar_ativo('ac induction motor 380 volts', top_k=1)
buscar_ativo('bearing failure motor com VSD', top_k=2)
buscar_ativo('explosion proof motor zona classificada', top_k=1)
buscar_ativo('thermal overload alarm exaustor', top_k=1)


Normalizacao: 790 variantes
  ac drive        -> Inversor de Frequência
  bearing failure -> Falha em Rolamento
  oc fault        -> Sobrecorrente
  thermal overload-> Sobrecarga Térmica

DEMONSTRACAO DO MOTOR DE BUSCA

----------------------------------------------------------------------
  Original   : "motor da area 3"
  Normalizada: "motor da area 3"
----------------------------------------------------------------------
  [1] MT-088  L2=15.0491 | Score=0.0623
      Área 1 - Exaustor do Forno
      Motor Inverter-Duty ABB
  [2] MT-015  L2=16.0572 | Score=0.0586
      Área 3 - Bomba de Drenagem Secundária
      Motor Monofásico Nidec

----------------------------------------------------------------------
  Original   : "ac induction motor 380 volts"
  Normalizada: "Motor de Indução Trifásico 380 volts"
     -> "ac induction motor" -> "Motor de Indução Trifásico"
----------------------------------------------------------------------
  [1] MT-042  L2=11.2875 | Score=0.0814
      Área 3

[{'tag': 'MT-042',
  'equipamento': 'Motor de Indução Trifásico WEG W22',
  'especificacao': '380V, 15kW, 1750 RPM, IP55, Classe F, Fator de Serviço 1.15',
  'localizacao': 'Área 3 - Bomba de Resfriamento Principal',
  'fabricante': 'WEG',
  'norma': 'IEC 60034 / NBR 17094',
  'contexto_busca': 'TAG: MT-042 | Equipamento: Motor de Indução Trifásico WEG W22 | Especificações: 380V, 15kW, 1750 RPM, IP55, Classe F, Fator de Serviço 1.15 | Localização: Área 3 - Bomba de Resfriamento Principal | Fabricante: WEG | Norma: IEC 60034 / NBR 17094',
  '_rank': 1,
  '_dist': 16.3252,
  '_score': 0.0577}]

## Célula 5 — Avaliação Rigorosa: 30 Consultas em 6 Categorias

**Motivação:** O feedback da Sprint 1 sinalizou a necessidade de testar com dados heterogêneos — onde ambiguidade, abreviações inconsistentes e textos multilíngues aparecem. Este conjunto é estruturado por nível de dificuldade crescente.

| Categoria | Nível | O que testa |
|---|---|---|
| A | Fácil | Localização coloquial |
| B | Médio | Especificação elétrica/mecânica com variações numéricas |
| C | Médio | Variante terminológica EN -> PT via `tabela_normalizacao.csv` |
| D | Difícil | Siglas técnicas, acrônimos e códigos de falha |
| E | Difícil | Múltiplos atributos potencialmente conflitantes |
| F | Muito Difícil | Code-switching PT + EN na mesma query |

**Métricas:** Precisão@2 (P), Recall@2 (R), F1, MRR (Mean Reciprocal Rank)


In [6]:
CONSULTAS = [
    # Cat A — Localizacao direta (Facil)
    ('motor da area 3',['MT-042','MT-015'],'A','Facil','Dois ativos na Area 3 — ambiguidade esperada'),
    ('equipamento no exaustor do forno',['MT-088'],'A','Facil','Localizacao tecnica especifica'),
    ('motor do compressor de gas natural',['MT-102'],'A','Facil','Localizacao por aplicacao'),
    ('motor do moinho',['MT-205'],'A','Facil','Localizacao por equipamento acoplado'),
    ('motor das bombas quimicas',['MT-411'],'A','Facil','Localizacao por setor'),
    # Cat B — Especificacao eletrica/mecanica (Medio)
    ('motor trifasico 380 volts 15kW',['MT-042'],'B','Medio','Especificacao eletrica em PT'),
    ('motor 440V alta temperatura classe H',['MT-088'],'B','Medio','Diferencia MT-088 de MT-411'),
    ('motor de altissima tensao 13 kilovolts',['MT-102'],'B','Medio','Variacao numerica sem unidade padrao'),
    ('motor monofasico baixa potencia 1.5kW',['MT-015'],'B','Medio','Tipo + potencia'),
    ('motor 690 volts rolamento isolado IE3',['MT-205'],'B','Medio','Tensao incomum + especificacao'),
    ('motor certificacao ATEX zona 1 IECEx',['MT-411'],'B','Medio','Norma area classificada'),
    # Cat C — Variante EN -> PT (Medio)
    ('ac induction motor 380 volts',['MT-042','MT-310'],'C','Medio','ac induction motor -> Motor de Inducao'),
    ('inverter duty motor forno',['MT-088'],'C','Medio','inverter duty -> motor com VSD'),
    ('squirrel cage motor pumping application',['MT-042','MT-310'],'C','Medio','EN puro — gaiola de esquilo bomba'),
    ('synchronous motor high voltage compressor',['MT-102'],'C','Medio','EN puro — sincrono alta tensao'),
    ('explosion proof motor classified area',['MT-411'],'C','Medio','explosion proof -> Motor a Prova de Explosao'),
    # Cat D — Abreviacao tecnica (Dificil)
    ('oc fault mt-088',['MT-088'],'D','Dificil','oc fault -> Sobrecorrente'),
    ('bearing failure motor area 1',['MT-088'],'D','Dificil','bearing failure + Area 1'),
    ('VSD motor 55kW forno',['MT-088'],'D','Dificil','VSD -> Inversor de Frequencia'),
    ('MCSA monitoramento motor com inversor',['MT-088','MT-205'],'D','Dificil','Tecnica de diagnostico — dois ativos'),
    ('DOL partida direta 220V',['MT-015'],'D','Dificil','DOL -> Partida Direta'),
    ('Y-delta starting pump motor',['MT-310'],'D','Dificil','Estrela-Triangulo -> bomba'),
    # Cat E — Consulta composta ambigua (Dificil)
    ('motor WEG ip55 com analise de vibracao',['MT-042','MT-205'],'E','Dificil','Dois WEG IP55 — ambos validos'),
    ('equipamento Siemens area 2 baixa rotacao',['MT-310'],'E','Dificil','MT-102 Siemens tem rotacao alta'),
    ('motor trifasico soft-starter nao WEG',['MT-088'],'E','Dificil','Negacao implicita — evitar MT-042'),
    ('motor mais potente da planta',['MT-102'],'E','Dificil','Superlativo — 500kW deve emergir'),
    # Cat F — Code-switching PT+EN (Muito Dificil)
    ('motor de inducao com bearing current problema',['MT-088','MT-205'],'F','Muito Dificil','PT+EN: bearing current em VSD'),
    ('thermal overload alarm area do forno',['MT-088'],'F','Muito Dificil','EN tecnico + PT geografico'),
    ('insulation class H motor running above rated temp',['MT-088'],'F','Muito Dificil','EN quase puro + classe H'),
    ('motor da Area 5 com IECEx certification preditiva',['MT-411'],'F','Muito Dificil','PT+EN + norma + manutencao'),
]

TOP_K = 2
CATS  = ['A','B','C','D','E','F']
rows  = []

for query, esperados, cat, nivel, desc in CONSULTAS:
    m     = buscar_ativo(query, top_k=TOP_K, verbose=False)
    ret   = [x['tag'] for x in m]
    s1    = m[0]['_score'] if m else 0.0
    acert = len(set(ret) & set(esperados))
    P     = round(acert/TOP_K, 3)
    R     = round(acert/len(esperados), 3)
    F1    = round(2*P*R/(P+R),3) if (P+R)>0 else 0.0
    mrr   = next((round(1/(i+1),3) for i,t in enumerate(ret) if t in esperados), 0.0)
    icon  = 'OK' if R==1.0 else ('PARCIAL' if R>0 else 'FALHA')
    rows.append({'cat':cat,'nivel':nivel,'desc':desc,'query':query,
                 'esperados':str(esperados),'retornados':str(ret),
                 'P':P,'R':R,'F1':F1,'MRR':mrr,'S1':s1,'icon':icon})

df_eval = pd.DataFrame(rows)

for cat in CATS:
    sub = df_eval[df_eval['cat']==cat]
    print(f"\n{'─'*80}")
    print(f"  Cat {cat} — {sub['nivel'].iloc[0]}")
    print(f"  P@{TOP_K}={sub['P'].mean():.3f}  R@{TOP_K}={sub['R'].mean():.3f}  "
          f"F1={sub['F1'].mean():.3f}  MRR={sub['MRR'].mean():.3f}")
    print(f"{'─'*80}")
    for _, r in sub.iterrows():
        print(f"  [{r['icon']}] {r['desc']}")
        print(f"    Query    : {r['query']}")
        print(f"    Esperado : {r['esperados']}")
        print(f"    Retornado: {r['retornados']}")
        print(f"    P={r['P']:.3f}  R={r['R']:.3f}  F1={r['F1']:.3f}  MRR={r['MRR']:.3f}  S1={r['S1']:.4f}")

print()
print('='*80)
print('METRICAS GLOBAIS')
print('='*80)
print(f"  Consultas avaliadas : {len(df_eval)}")
print(f"  Precisao Media @{TOP_K}  : {df_eval['P'].mean():.3f}")
print(f"  Recall Medio @{TOP_K}    : {df_eval['R'].mean():.3f}")
print(f"  F1-Score Medio      : {df_eval['F1'].mean():.3f}")
print(f"  MRR Medio           : {df_eval['MRR'].mean():.3f}")
n_ok = (df_eval['R']==1.0).sum()
print(f"  Recall Perfeito     : {n_ok}/{len(df_eval)} ({100*n_ok/len(df_eval):.1f}%)")
print()
print(f"  {'Cat':>4}  {'Nivel':<14}  {'N':>3}  {'P@2':>6}  {'R@2':>6}  {'F1':>6}  {'MRR':>6}")
print('  ' + '-'*55)
for cat in CATS:
    sub = df_eval[df_eval['cat']==cat]
    print(f"  {cat:>4}  {sub['nivel'].iloc[0]:<14}  {len(sub):>3}  "
          f"{sub['P'].mean():>6.3f}  {sub['R'].mean():>6.3f}  "
          f"{sub['F1'].mean():>6.3f}  {sub['MRR'].mean():>6.3f}")

falhas = df_eval[df_eval['R']<1.0]
print()
print(f'ANALISE DE ERROS ({len(falhas)} falhas):')
print('-'*80)
if len(falhas)==0:
    print('  Recall perfeito em todas as consultas.')
else:
    for _, r in falhas.iterrows():
        print(f"  FALHA [{r['cat']}/{r['nivel']}] {r['query']}  S1={r['S1']:.4f}")
        print(f"    Esperado : {r['esperados']}")
        print(f"    Retornado: {r['retornados']}")
        print(f"    Diagnostico: verificar tabela_normalizacao.csv ou enriquecer contexto_busca.")

df_eval.to_csv(PATH_METRICAS_CSV, index=False, encoding='utf-8')
print(f'\nMetricas salvas: {PATH_METRICAS_CSV}')



────────────────────────────────────────────────────────────────────────────────
  Cat A — Facil
  P@2=0.400  R@2=0.700  F1=0.500  MRR=0.600
────────────────────────────────────────────────────────────────────────────────
  [PARCIAL] Dois ativos na Area 3 — ambiguidade esperada
    Query    : motor da area 3
    Esperado : ['MT-042', 'MT-015']
    Retornado: ['MT-088', 'MT-015']
    P=0.500  R=0.500  F1=0.500  MRR=0.500  S1=0.0623
  [OK] Localizacao tecnica especifica
    Query    : equipamento no exaustor do forno
    Esperado : ['MT-088']
    Retornado: ['MT-042', 'MT-088']
    P=0.500  R=1.000  F1=0.667  MRR=0.500  S1=0.0635
  [OK] Localizacao por aplicacao
    Query    : motor do compressor de gas natural
    Esperado : ['MT-102']
    Retornado: ['MT-102', 'MT-042']
    P=0.500  R=1.000  F1=0.667  MRR=1.000  S1=0.0497
  [FALHA] Localizacao por equipamento acoplado
    Query    : motor do moinho
    Esperado : ['MT-205']
    Retornado: ['MT-042', 'MT-102']
    P=0.000  R=0.000  F1=

## Célula 6 — Entregável 3: NLG Operacional — Templates e Documentação

### 📄 Templates de Frase — Regras de Composição Textual

A geração de linguagem natural (NLG) segue uma arquitetura **Template-Based com lógica condicional**:

```
[dados estáticos do ativo]  +  [telemetria IoT em tempo real]  →  [relatório em linguagem natural]
  (DataFrame Digital Twin)       (sensores: V, A, RPM, °C)          (templates parametrizados)
```

#### Template Principal — Relatório Operacional

```
{ÍCONE} RELATÓRIO OPERACIONAL AUTOMÁTICO | Ativo: {TAG}
{LINHA_SEPARADORA}
O equipamento {NOME_EQUIPAMENTO}, localizado em '{LOCALIZAÇÃO}',
encontra-se atualmente {ESTADO_OPERACIONAL}.

Última leitura dos sensores industriais:
  • Tensão       : {TENSÃO} V
  • Corrente     : {CORRENTE} A
  • Rotação      : {RPM} RPM
  • Temp. Carcaça: {TEMPERATURA} °C

Diretriz: {DIRETRIZ}
```

#### Regras de Composição — Estado Operacional

| Condição | `{ESTADO_OPERACIONAL}` | `{DIRETRIZ}` | Ícone |
|---|---|---|---|
| Todos parâmetros dentro do nominal | `operando dentro dos parâmetros normais estabelecidos` | `Nenhuma ação corretiva é necessária no momento.` | ✅ |
| Temperatura > limiar classe térmica | `apresentando desvio operacional crítico (Alarme de Temperatura)` | `Recomenda-se abertura de ordem de serviço e inspeção imediata.` | 🚨 |
| Corrente > 110% nominal | `apresentando sobrecorrente acima do valor nominal` | `Verificar carga acoplada e condições de acionamento.` | ⚠️ |
| Sem dados de telemetria | — | — | ⚠️ mensagem padrão |

#### Template Curto — Interface de Visualização (Dashboard)

```
{TAG} | {STATUS_EMOJI} {STATUS_LABEL} | {TENSÃO}V · {CORRENTE}A · {RPM}RPM · {TEMPERATURA}°C
```

**Exemplo:** `MT-042 | ✅ Normal | 379V · 14.5A · 1745RPM · 45°C`


In [8]:
# ==============================================================================
# CÉLULA 6: ENRIQUECIMENTO TEXTUAL (NLG OPERACIONAL)
# Entregável 3: Templates de frase + saídas textuais para a interface
# ==============================================================================

#  Mock de telemetria IoT 
# Em produção, estes dados viriam de sensores via MQTT/OPC-UA em tempo real.
telemetria_mock = {
    'MT-042': {'tensao': 379,   'corrente': 14.5,  'rpm': 1745, 'temp': 45,  'vib': 1.2, 'status': 'Normal'},
    'MT-088': {'tensao': 440,   'corrente': 85.0,  'rpm': 3450, 'temp': 92,  'vib': 4.8, 'status': 'Alarme de Temperatura'},
    'MT-015': {'tensao': 218,   'corrente':  5.2,  'rpm': 3590, 'temp': 40,  'vib': None,'status': 'Normal'},
    'MT-205': {'tensao': 688,   'corrente': 220.0, 'rpm': 1198, 'temp': 78,  'vib': 5.3, 'status': 'Vibracao Elevada'},
    'MT-310': {'tensao': 381,   'corrente': 38.0,  'rpm': 1448, 'temp': 55,  'vib': 2.1, 'status': 'Normal'},
    'MT-411': {'tensao': 439,   'corrente': 42.0,  'rpm': 1749, 'temp': 68,  'vib': 1.7, 'status': 'Normal'},
    # MT-102 sem telemetria (offline)
}

#  Dicionário de regras de estado 
REGRAS_ESTADO = {
    'Normal': {
        'icone'   : '✅',
        'estado'  : 'operando dentro dos parâmetros normais estabelecidos',
        'diretriz': 'Nenhuma ação corretiva é necessária no momento.',
    },
    'Alarme de Temperatura': {
        'icone'   : '🚨',
        'estado'  : 'apresentando um desvio operacional crítico (Alarme de Temperatura)',
        'diretriz': 'Recomenda-se a abertura de uma ordem de serviço e inspeção imediata.',
    },
    'Sobrecorrente': {
        'icone'   : '⚠️',
        'estado'  : 'apresentando sobrecorrente acima do valor nominal',
        'diretriz': 'Verificar carga acoplada, couplings e condições de acionamento.',
    },
    'Vibracao Elevada': {
        'icone'   : '⚠️',
        'estado'  : 'com vibração acima do limiar ISO 20816-3 Zona B (> 4,5 mm/s RMS)',
        'diretriz': 'Agendar análise espectral de vibração. Verificar balanceamento e alinhamento.',
    },
}


#  Template 1: Relatório Operacional Completo 
def gerar_relatorio_completo(tag_ativo: str) -> str:
    """
    NLG baseado em templates: cruza dados estáticos (Digital Twin)
    com variáveis dinâmicas (telemetria IoT).
    Destino: documento de manutenção / notificação ao operador.
    """
    if tag_ativo not in telemetria_mock:
        return (
            f'⚫ RELATÓRIO OPERACIONAL | Ativo: {tag_ativo}\n'
            f'{"─" * 56}\n'
            f'Sem dados de telemetria para {tag_ativo}. Verificar conectividade IIoT.\n'
        )

    ativo = df_ativos[df_ativos['tag'] == tag_ativo].iloc[0]
    dados = telemetria_mock[tag_ativo]
    regra = REGRAS_ESTADO.get(dados['status'], REGRAS_ESTADO['Normal'])

    # Campo condicional: vibração só aparece se sensor presente
    linha_vib = (
        f"  • Vibração     : {dados['vib']} mm/s RMS\n"
        if dados.get('vib') is not None else ''
    )

    relatorio = (
        f"{regra['icone']} RELATÓRIO OPERACIONAL AUTOMÁTICO | Ativo: {tag_ativo}\n"
        f"{'─' * 56}\n"
        f"O equipamento {ativo['equipamento']}, localizado em '{ativo['localizacao']}', "
        f"encontra-se atualmente {regra['estado']}.\n\n"
        f"Última leitura dos sensores industriais:\n"
        f"  • Tensão       : {dados['tensao']} V\n"
        f"  • Corrente     : {dados['corrente']} A\n"
        f"  • Rotação      : {dados['rpm']} RPM\n"
        f"  • Temp. Carcaça: {dados['temp']} °C\n"
        f"{linha_vib}"
        f"\nDiretriz: {regra['diretriz']}\n"
    )

    path_out = os.path.join(DIR_RELATORIOS, f'relatorio_{tag_ativo}.txt')
    with open(path_out, 'w', encoding='utf-8') as f:
        f.write(relatorio)

    return relatorio


#  Template 2: Linha de Dashboard (formato compacto) 
def gerar_linha_dashboard(tag_ativo: str) -> str:
    """
    Formato compacto para HMI / SCADA / interface web.
    Substitui dados brutos por linguagem compreensível ao operador.
    """
    if tag_ativo not in telemetria_mock:
        return f'  {tag_ativo:<8} | ⚫ OFFLINE                | Sem telemetria'

    dados = telemetria_mock[tag_ativo]
    icone = REGRAS_ESTADO.get(dados['status'], REGRAS_ESTADO['Normal'])['icone']
    vib_s = f" · {dados['vib']} mm/s" if dados.get('vib') is not None else ''

    return (
        f"  {tag_ativo:<8} | {icone} {dados['status']:<22} | "
        f"{dados['tensao']:>4}V · {dados['corrente']:>5.1f}A · "
        f"{dados['rpm']:>4}RPM · {dados['temp']:>2}°C{vib_s}"
    )


# Template 3: Resposta Conversacional (NLP → NLG) 
def gerar_resposta_consulta(query: str) -> str:
    """
    Combina busca semântica com NLG para responder ao operador.
    Fluxo: query → buscar_ativo() → template → linguagem natural
    Destino: chatbot / assistente virtual do Digital Twin.
    """
    resultados = buscar_ativo(query, top_k=1, verbose=False)   # ← nome correto
    if not resultados:
        return 'Nenhum ativo encontrado para a consulta informada.'

    tag   = resultados[0]['tag']
    score = resultados[0]['_score']

    if tag in telemetria_mock:
        dados = telemetria_mock[tag]
        regra = REGRAS_ESTADO.get(dados['status'], REGRAS_ESTADO['Normal'])
        vib_s = f" · {dados['vib']} mm/s" if dados.get('vib') is not None else ''
        return (
            f"Ativo identificado: {tag} (similaridade: {score:.3f})\n"
            f"→ {resultados[0]['equipamento']} | {resultados[0]['localizacao']}\n"
            f"→ Estado atual: {regra['icone']} {dados['status']} | "
            f"{dados['tensao']}V · {dados['corrente']}A · {dados['rpm']}RPM · {dados['temp']}°C{vib_s}\n"
            f"→ Diretriz: {regra['diretriz']}"
        )
    else:
        return (
            f"Ativo identificado: {tag} (similaridade: {score:.3f})\n"
            f"→ {resultados[0]['equipamento']} | {resultados[0]['localizacao']}\n"
            f"→ Estado atual: ⚫ OFFLINE — sem dados de telemetria disponíveis."
        )


# Geração das saídas 
print('=' * 65)
print('📄 TEMPLATE 1 — RELATÓRIOS OPERACIONAIS')
print('=' * 65)
print()
for tag in ['MT-042', 'MT-088', 'MT-205', 'MT-102']:
    print(gerar_relatorio_completo(tag))

print()
print('=' * 65)
print('🖥️  TEMPLATE 2 — PAINEL DE SUPERVISÃO (Dashboard)')
print('=' * 65)
print()
print(f'  {"TAG":<8} | {"STATUS":<24} | LEITURAS')
print('  ' + '─' * 65)
for tag in df_ativos['tag'].tolist():
    print(gerar_linha_dashboard(tag))

print()
print()
print('=' * 65)
print('💬 TEMPLATE 3 — RESPOSTA CONVERSACIONAL (NLP → NLG)')
print('=' * 65)
print()
for q in [
    'motor da área 3',
    'alarm condition no exaustor',
    'motor de alta tensão compressor',
    'motor com vibracao elevada moinho',
]:
    print(f'Operador: "{q}"')
    print(f'Sistema : {gerar_resposta_consulta(q)}')
    print()

📄 TEMPLATE 1 — RELATÓRIOS OPERACIONAIS

✅ RELATÓRIO OPERACIONAL AUTOMÁTICO | Ativo: MT-042
────────────────────────────────────────────────────────
O equipamento Motor de Indução Trifásico WEG W22, localizado em 'Área 3 - Bomba de Resfriamento Principal', encontra-se atualmente operando dentro dos parâmetros normais estabelecidos.

Última leitura dos sensores industriais:
  • Tensão       : 379 V
  • Corrente     : 14.5 A
  • Rotação      : 1745 RPM
  • Temp. Carcaça: 45 °C
  • Vibração     : 1.2 mm/s RMS

Diretriz: Nenhuma ação corretiva é necessária no momento.

🚨 RELATÓRIO OPERACIONAL AUTOMÁTICO | Ativo: MT-088
────────────────────────────────────────────────────────
O equipamento Motor Inverter-Duty ABB, localizado em 'Área 1 - Exaustor do Forno', encontra-se atualmente apresentando um desvio operacional crítico (Alarme de Temperatura).

Última leitura dos sensores industriais:
  • Tensão       : 440 V
  • Corrente     : 85.0 A
  • Rotação      : 3450 RPM
  • Temp. Carcaça: 92 °C
 

## Célula 7 — Relatório de Integração e Artefatos Entregues

In [9]:
# ==============================================================================
# CÉLULA 7: RELATÓRIO FINAL — INTEGRAÇÃO SPRINT 1 + SPRINT 2
# ==============================================================================

import datetime

# ── Gerar documento Markdown com templates e regras ────────────────────────────
templates_doc = '''# Templates NLG — Sprint 2 | PLN Forzy
## FIAP x Forzy | Processamento de Linguagem Natural
Data: ''' + datetime.date.today().isoformat() + '''

---

## Visão Geral da Arquitetura NLG

A geração de linguagem natural (NLG) da Sprint 2 segue uma arquitetura
**Template-Based com lógica condicional**, que cruza:

- **Dados estáticos** (Digital Twin): cadastro do ativo (TAG, equipamento, especificação, localização)
- **Dados dinâmicos** (Telemetria IIoT): leituras de sensores (V, A, RPM, °C, status)

---

## Template 1: Relatório Operacional Completo

**Destino:** Documento de manutenção, notificação ao operador, ordem de serviço.

```
{ÍCONE} RELATÓRIO OPERACIONAL AUTOMÁTICO | Ativo: {TAG}
{LINHA_SEPARADORA}
O equipamento {NOME_EQUIPAMENTO}, localizado em '{LOCALIZAÇÃO}',
encontra-se atualmente {ESTADO_OPERACIONAL}.

Última leitura dos sensores industriais:
  • Tensão       : {TENSÃO} V
  • Corrente     : {CORRENTE} A
  • Rotação      : {RPM} RPM
  • Temp. Carcaça: {TEMPERATURA} °C

Diretriz: {DIRETRIZ}
```

### Regras de Composição — Campo {ESTADO_OPERACIONAL}

| Status Sensor | {ÍCONE} | {ESTADO_OPERACIONAL} | {DIRETRIZ} |
|---|---|---|---|
| Normal | ✅ | operando dentro dos parâmetros normais estabelecidos | Nenhuma ação corretiva é necessária no momento. |
| Alarme de Temperatura | 🚨 | apresentando um desvio operacional crítico (Alarme de Temperatura) | Recomenda-se a abertura de uma ordem de serviço e inspeção imediata. |
| Sobrecorrente | ⚠️ | apresentando sobrecorrente acima do valor nominal | Verificar carga acoplada, couplings e condições de acionamento. |
| Offline | ⚫ | — | Verificar conectividade do sensor IIoT. |

---

## Template 2: Linha de Dashboard (Formato Compacto)

**Destino:** Painéis de supervisão HMI / SCADA / interface web do Digital Twin.

```
{TAG} | {EMOJI} {STATUS_LABEL} | {TENSÃO}V · {CORRENTE}A · {RPM}RPM · {TEMPERATURA}°C
```

**Exemplos de saída:**
- `MT-042 | ✅ Normal                 |  379V ·  14.5A · 1745RPM · 45°C`
- `MT-088 | 🔴 Alarme de Temperatura  |  440V ·  85.0A · 3450RPM · 92°C`
- `MT-102 | ⚫ OFFLINE               | Sem telemetria`

**Princípio de design:** substituir dados brutos por linguagem contextualizada.
O operador não precisa memorizar limiares — o sistema traduz o estado para uma
leitura imediata.

---

## Template 3: Resposta Conversacional (NLP → NLG)

**Destino:** Interface de chatbot / assistente virtual do Digital Twin.

```
Ativo identificado: {TAG} (similaridade: {SCORE})
→ {NOME_EQUIPAMENTO} | {LOCALIZAÇÃO}
→ Estado atual: {ÍCONE} {STATUS} | {TENSÃO}V · {CORRENTE}A · {RPM}RPM · {TEMPERATURA}°C
→ Diretriz: {DIRETRIZ}
```

**Exemplo de uso:**
- Operador: "motor da área 3"
- Sistema: "Ativo identificado: MT-042 (similaridade: 0.812)
  → Motor de Indução Trifásico WEG W22 | Área 3 - Bomba de Resfriamento Principal
  → Estado atual: ✅ Normal | 379V · 14.5A · 1745RPM · 45°C
  → Diretriz: Nenhuma ação corretiva é necessária no momento."

---

## Integração com a Sprint 1

| Artefato Sprint 1 | Uso na Sprint 2 |
|---|---|
| `glossario_tecnico_motores_v5.csv` | Vocabulário técnico de referência para avaliação |
| `tabela_normalizacao.csv` (790 variantes) | Pipeline de normalização terminológica da query |
| `corpus_consolidado_v1.csv` | Base textual para validação do corpus |
| `corpus_stats.json` | Métricas de vocabulário usadas para justificar o modelo |
| `stopwords_tecnicas.txt` | Referência de termos a preservar no pré-processamento |

---

*Gerado automaticamente pelo pipeline Sprint 2 | PLN Forzy*
'''

with open(PATH_TEMPLATES_MD, 'w', encoding='utf-8') as f:
    f.write(templates_doc)

# ── Sumário final ─────────────────────────────────────────────────────────────
print('=' * 65)
print('📦 ARTEFATOS ENTREGUES — SPRINT 2')
print('=' * 65)
print()
print('📁 Entregável 1 — Representação Vetorial:')
print(f'   ✅ Embeddings gerados: {indice_faiss.ntotal} vetores × 384 dims (float32)')
print(f'   ✅ Índice FAISS: IndexFlatL2 (busca exata, determinístico)')
print(f'   ✅ Modelo: paraphrase-multilingual-MiniLM-L12-v2')
print(f'   💾 Stats: {PATH_STATS_JSON}')
print()
print('📁 Entregável 2 — Módulo de Consulta Semântica:')
print(f'   ✅ buscar_ativo_por_nlp() com normalização terminológica (790 variantes)')
print(f'   ✅ 15 consultas-teste avaliadas com P@2, R@2 e F1')
print(f'   💾 Métricas: {PATH_METRICAS_CSV}')
print()
print('📁 Entregável 3 — Enriquecimento Textual NLG:')
print('   ✅ Template 1: Relatório Operacional Completo')
print('   ✅ Template 2: Linha de Dashboard compacta')
print('   ✅ Template 3: Resposta Conversacional (NLP → NLG)')
print(f'   💾 Documentação: {PATH_TEMPLATES_MD}')
print()
print('📁 Integração com Sprint 1:')
print('   ✅ tabela_normalizacao.csv → pipeline de query')
print('   ✅ corpus_stats.json → justificativa do modelo')
print('   ✅ glossario_tecnico_motores_v5 → referência vocabular')
print()
print('=' * 65)
print('✅ SPRINT 2 CONCLUÍDA COM SUCESSO')
print('=' * 65)


📦 ARTEFATOS ENTREGUES — SPRINT 2

📁 Entregável 1 — Representação Vetorial:
   ✅ Embeddings gerados: 7 vetores × 384 dims (float32)
   ✅ Índice FAISS: IndexFlatL2 (busca exata, determinístico)
   ✅ Modelo: paraphrase-multilingual-MiniLM-L12-v2
   💾 Stats: c:\Users\gabri\Downloads\sprint2_PLN\sprint2\relatorios\corpus_stats_sprint2.json

📁 Entregável 2 — Módulo de Consulta Semântica:
   ✅ buscar_ativo_por_nlp() com normalização terminológica (790 variantes)
   ✅ 15 consultas-teste avaliadas com P@2, R@2 e F1
   💾 Métricas: c:\Users\gabri\Downloads\sprint2_PLN\sprint2\relatorios\metricas_avaliacao_sprint2.csv

📁 Entregável 3 — Enriquecimento Textual NLG:
   ✅ Template 1: Relatório Operacional Completo
   ✅ Template 2: Linha de Dashboard compacta
   ✅ Template 3: Resposta Conversacional (NLP → NLG)
   💾 Documentação: c:\Users\gabri\Downloads\sprint2_PLN\sprint2\relatorios\templates_nlg_sprint2.md

📁 Integração com Sprint 1:
   ✅ tabela_normalizacao.csv → pipeline de query
   ✅ corpus_stats